# MaxEnt IRL Driving Simulation Pipeline

This notebook performs the complete workflow for a driving behavior imitation learning pipeline:

- Preprocess raw trajectory data
- Build trajectory sequences
- Construct the MDP representation
- Train a Maximum Entropy Inverse Reinforcement Learning (MaxEnt IRL) model
- Evaluate the learned policy
- Save artifacts for visualization and later analysis
- Visualisation of graphs
- Simulation in highway-env, python gymnasium 


The code is divided into steps, skip steps 1 - 10 if Agent is already trained (irl_artifacts.pkl exists)

In [ ]:

# Imports

# Custom project modules
import preprocess as pp
import build_trajectories2 as build
import DrivEnv as env
import maxEntIRL as me
import MDPBuilder as mdp

# Standard libraries
import os
import random
import pickle

# Data science libraries
import pandas as pd
import numpy as np

## Step 1 — Define File Paths

These paths define:
- The raw driving dataset
- The preprocessed dataset output
- The generated trajectory dataset


In [ ]:
# Raw dataset path
raw_data_path = r"data\M40-d07-h08.dat"

# Preprocessed CSV output
preprocessed_output_path = r"data\dataset.csv"

# Generated trajectories output
trajectories_output_path = r"data\M40_d07_h08_trajectories_updated.csv"

## Step 2 — Preprocess Raw Data

This step converts the raw `.dat` traffic dataset into a cleaned CSV format suitable for trajectory extraction.

> Uncomment the preprocessing line if preprocessing has not been done yet.


In [ ]:

# Preprocess the raw traffic data


# Uncomment this line if dataset.csv does not already exist
# pp.Preprocess(raw_data_path, preprocessed_output_path)

print("Preprocessing step completed.")

## Step 3 — Build Trajectories

This step converts the preprocessed dataset into trajectory sequences that can be used by the MDP and IRL pipeline.


In [ ]:

# Build trajectories from preprocessed data


builder = build.BuildTrajectories(
    preprocessed_output_path,
    trajectories_output_path
)

# Generate trajectory dataframe
trajectories_df = builder.run(trajectories_output_path)

# Uncomment if Alternative: Load directly if trajectories already exist
# trajectories_df = pd.read_csv(trajectories_output_path)

print("Trajectory generation completed.")
print(f"Number of rows: {len(trajectories_df)}")

## Step 4 — Create the Environment and Build the MDP

Here we:
- Create the driving environment
- Build the Markov Decision Process (MDP)
- Generate transition matrices and feature representations


In [ ]:

# Create driving environment


env_instance = env.DrivingEnv(trajectories_df)


# Build the MDP


mdp_builder = mdp.MDPBuilder(trajectories_df)
mdp_data = mdp_builder.build_all()

print("MDP construction completed.")
print(f"Number of states: {mdp_data['n_states']}")
print(f"Number of actions: {mdp_data['n_actions']}")

## Step 5 — Train/Test Split

We split the expert trajectories into:
- 80% training data
- 20% testing data

The split is randomized using a fixed seed for reproducibility.


In [ ]:

# Train/Test Split


all_trajectories = mdp_data["expert_trajectories"]

# Shuffle for random distribution
random.seed(42)
random.shuffle(all_trajectories)

# 80/20 split
split_idx = int(0.8 * len(all_trajectories))

train_trajectories = all_trajectories[:split_idx]
test_trajectories = all_trajectories[split_idx:]

print(f"Data Split: {len(train_trajectories)} Train | {len(test_trajectories)} Test")

# Recompute initial state distribution and horizon for training set
initial_dist, train_horizon = mdp_builder.get_initial_dist_and_horizon(train_trajectories)

print(f"Training Horizon: {train_horizon}")

## Step 6 — Initialize and Train the MaxEnt IRL Agent

The Maximum Entropy Inverse Reinforcement Learning algorithm learns a reward function from expert demonstrations.

Inputs include:
- Transition probabilities
- Feature matrix
- Initial state distribution
- Expert trajectories


In [ ]:

# Initialize MaxEnt IRL Agent


print("Initializing MaxEnt IRL...")

irl_agent = me.MaxEntIRL(
    n_states=mdp_data["n_states"],
    n_actions=mdp_data["n_actions"],
    transition_probs=mdp_data["transition_matrix"],
    feature_matrix=mdp_data["feature_matrix"],
    initial_state_dist=initial_dist,
    horizon=train_horizon,
    learning_rate=0.01
)

print("Starting MaxEnt IRL Training Loop...")

# Remove trajectory IDs before training
just_train_trajs = [traj for traj_id, traj in train_trajectories]

# Train the IRL model
learned_theta, learned_policy = irl_agent.train(
    expert_trajectories=just_train_trajs,
    iterations=60
)

print("Training completed.")
print("Learned Reward Weights:")
print(learned_theta)

## Step 7 — Save Artifacts for Visualization

This section saves:
- Expert trajectories
- Simulated trajectories
- Learned reward parameters
- Policy and MDP artifacts

These outputs can later be used for plotting and visualization.


In [ ]:

# Create results directory


os.makedirs("results", exist_ok=True)

print("Results directory ready.")

## Step 8 — Generate Sample Expert and Simulated Trajectories

We select one test trajectory and generate:
- The expert trajectory
- A simulated trajectory from the learned policy

These will be saved as CSV files for analysis.


In [ ]:

# Generate sample trajectories for visualization


# Select a sample trajectory from the test set
sample_traj_id, sample_expert_traj = test_trajectories[0]

# Determine trajectory horizon
horizon = len(sample_expert_traj)

# Extract expert trajectory dataframe
sample_expert_df = env_instance.data[
    env_instance.data["traj_id"] == sample_traj_id
].head(horizon).reset_index(drop=True)

# Simulate trajectory using learned policy
sample_sim_df = env_instance.simulate_agent(
    mdp_builder,
    learned_policy,
    sample_traj_id,
    max_steps=horizon
)

print("Sample trajectories generated.")

## Step 9 — Save CSV Outputs

Save the expert and simulated trajectories for easy inspection and visualization.


In [ ]:

# Save trajectory CSV files


sample_expert_df.to_csv("results/expert_trajectory.csv", index=False)
sample_sim_df.to_csv("results/simulated_trajectory.csv", index=False)

print("Trajectory CSV files saved.")

## Step 10 — Save Model Artifacts

Store all important training artifacts in a pickle file.

Saved contents:
- Learned reward weights (`theta`)
- Learned policy
- MDP data
- Test trajectories


In [ ]:

# Save artifacts using pickle


artifacts = {
    "theta": learned_theta,
    "policy": learned_policy,
    "mdp_data": mdp_data,
    "test_trajectories": test_trajectories
}

with open("results/irl_artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("All artifacts saved successfully in the 'results/' folder.")

## Step 11 — Evaluate the Learned Policy

We now evaluate the learned policy using the test trajectories.

The environment compares the simulated behavior against expert demonstrations.


In [ ]:

# Evaluate learned policy


print("Starting Simulation & Evaluation on Test Set...")

env_instance.evaluate_performance(
    mdp_builder=mdp_builder,
    policy=learned_policy,
    test_trajectories=test_trajectories
)

print("Evaluation completed.")

# Visualization Pipeline

This section loads the saved artifacts and visualizes:

- Expert trajectories
- Simulated trajectories
- Learned reward behavior
- Policy behavior
- Experiment 2 style comparisons (optional)


In [4]:
import pickle
import pandas as pd
import Visualiser as vis
import numpy as np
import os

Toggle between Visualise ouput of Experiment 2 or not

In [5]:
# Toggle between:
# False -> Standard IRL visualization
# True  -> Experiment 2 style comparison visualization
experiment2 = False

In [ ]:

# Visualization Main Execution Block

# Directory where all saved outputs are stored
results_dir = "results"

print("Loading saved artifacts...")


# Experiment 2 Visualization Mode


if experiment2:
    print("Running Experiment 2 Visualizations...")

    # Import aggregated and conservative driving styles
    # along with policies and expert demonstrations
    agg_sim, cons_sim, visualizer, agg_policy, cons_policy, agg_exp, cons_exp = vis.import_data_for_styles()

    # Optional: Run full visualization pipeline
    # visualizer.run_all(
    #     expert_df=agg_sim,
    #     simulated_df=cons_sim,
    #     test_trajectories=0,
    #     policy=agg_policy,
    #     experiment2=True
    # )

    # Plot comparison between aggressive and conservative styles
    visualizer.plot_style_comparison(
        agg_sim,
        cons_sim,
        agg_exp,
        cons_exp
    )


# Standard Visualization Mode


else:

    try:
        
        # Load saved trajectory CSV files
        

        expert_df = pd.read_csv(f"{results_dir}/expert_trajectory.csv")
        sim_df = pd.read_csv(f"{results_dir}/simulated_trajectory.csv")

        print("Trajectory CSV files loaded successfully.")

        
        # Load saved IRL artifacts
        

        with open(f"{results_dir}/irl_artifacts.pkl", "rb") as f:
            artifacts = pickle.load(f)

        print("IRL artifacts loaded successfully.")

        
        # Extract stored objects
        

        theta = artifacts["theta"]
        mdp_data = artifacts["mdp_data"]
        test_trajectories = artifacts["test_trajectories"]

        # Policy may not exist in older artifact files
        policy = artifacts.get("policy", None)

        
        # Initialize the visualizer

        visualizer = vis.IRLVisualizer(
            mdp_data=mdp_data,
            theta=theta
        )

        print("Visualizer initialized.")

        
        # Run the visualization pipeline
        

        visualizer.run_all(
            expert_df=expert_df,
            simulated_df=sim_df,
            test_trajectories=test_trajectories,
            policy=policy,
            experiment2=False
        )

        print("Visualization pipeline completed successfully.")

    except FileNotFoundError:
        print("Error: Could not find the saved files.")
        print("Make sure the training notebook has been executed first.")

# Highway Environment IRL Simulation

This notebook section runs the trained MaxEnt IRL policy inside the `highway-env` simulator.

The simulator:
- Loads trained IRL policies
- Simulates driving episodes
- Compares behavior against expert trajectories
- Computes driving statistics
- Saves evaluation metrics for later analysis

Metrics collected:
- Average speed
- Number of lane changes
- Crash count


 Necessary imports for simulation

In [2]:
import pickle
import pandas as pd
import numpy as np
import time
import Simulator as simHigh
from highway_env.vehicle.kinematics import Vehicle


Load artifacts and initialise

In [3]:

# Main Simulation Execution
# Load Style Artifacts


with open("results/styles_artifacts.pkl", "rb") as f:
    styles = pickle.load(f)

# Aggressive driving profile
agg_mdp_data = styles["aggressive"]["mdp_data"]
agg_policy = styles["aggressive"]["policy"]

# Conservative driving profile
cons_mdp_data = styles["conservative"]["mdp_data"]
cons_policy = styles["conservative"]["policy"]


# Load Default IRL Artifacts


print("Loading IRL Artifacts and Expert Data...")
try:
    with open("results/irl_artifacts.pkl", "rb") as f:
        artifacts = pickle.load(f)
            
        mdp_data = artifacts["mdp_data"]
        policy = artifacts["policy"]
        expert_df = pd.read_csv("results/expert_trajectory.csv")
        
except FileNotFoundError:
    print(" Error: Missing files in 'results/' folder.")
    exit()

print("Initializing Unified Simulator...")
sim = simHigh.IntegratedHighwaySimulator(mdp_data=mdp_data, render=True)
    
state_idx = sim.reset()
done = truncated = False
    
# 1. EXTRAER EL INICIO DEL EXPERTO
expert_start_x = expert_df.iloc[0]['x']
expert_start_y = expert_df.iloc[0]['y']
expert_start_v = expert_df.iloc[0]['computed_v']

# 2. FUNCIÓN DE CONVERSIÓN DE CARRILES (De M40 España a Highway-Env)
# M40: 500, 503, 506 (Ancho de 3m) -> Índices: 0, 1, 2 -> Highway-Env: 0, 4, 8 (Ancho de 4m)
def normalize_y(y_real):
    # Usamos tu lógica adaptada: restamos 500, dividimos por 3 para sacar el carril, y escalamos a 4m
    lane_idx = round((y_real - 500.0) / 3.0)
    # Asegurarnos de que no se salga de los 3 carriles (0, 1, o 2)
    lane_idx = max(0, min(2, lane_idx)) 
    return lane_idx * 4.0
sim_start_y = normalize_y(expert_start_y)

# 3. FORZAR AL COCHE EGO (VERDE) A EMPEZAR EXACTAMENTE DONDE ESTABA EL HUMANO
# Ya no empieza aleatorio, empieza en el carril exacto y a la velocidad exacta
sim.env.unwrapped.vehicle.position = np.array([0.0, sim_start_y])
sim.env.unwrapped.vehicle.speed = expert_start_v

# 4. CREAR EL COCHE FANTASMA EN EL EXACTO MISMO PÍXEL
ghost_car = Vehicle(
    road=sim.env.unwrapped.road, 
    position=[0.0, sim_start_y], 
    speed=expert_start_v
)
ghost_car.color = (255, 0, 255) # Magenta
ghost_car.crashed = False 
#sim.env.unwrapped.road.vehicles.append(ghost_car)

print("\n--- Starting Ghost Car Comparison (Shadow Mode) ---")
step_count = 0
speed = 0

Loading IRL Artifacts and Expert Data...
Initializing Unified Simulator...

--- Starting Ghost Car Comparison (Shadow Mode) ---


In [4]:
# Only run to compare with expert trajectory
sim.env.unwrapped.road.vehicles.append(ghost_car)

Run episode

In [5]:
# Episode Rollout

while not truncated and not done:
    # Ask IRL Agent for Action
    action_probs = policy[state_idx]
    prob_sum = np.sum(action_probs)
    #if prob_sum > 0:
    #    action_probs = action_probs / prob_sum
    #else:
    #    # PROFESSIONAL FALLBACK: Maintain current trajectory rather than panicking arbitrarily
    #    action_probs = np.zeros(len(action_probs))
    #    idle_action_idx = sim.physical_actions.index((0.0, 0)) # Find the (0 delta_v, 0 delta_lane) action
    #    action_probs[idle_action_idx] = 1.0
    action_idx = np.random.choice(len(action_probs), p=action_probs)
    #print(f"Step {step_count}: State {state_idx} -> Action {action_idx}: {sim.physical_actions[action_idx]} (Action Probabilities: {action_probs})")
    
    # Step Simulator
    next_state_idx, reward, done, truncated, info = sim.step(action_idx)
    
    sim.env.unwrapped.vehicle.crashed = False
    #ghost_car.crashed = False
    
    if step_count < len(expert_df):
        historical_data = expert_df.iloc[step_count]
        
        # Avanzar X relativo al punto de partida original
        normalized_x = historical_data['x'] - expert_start_x
        # Calcular Y con nuestra nueva función de carriles
        normalized_y = normalize_y(historical_data['y'])
        
        #ghost_car.position = np.array([normalized_x, normalized_y])
        #ghost_car.speed = historical_data['computed_v']
        
    speed += sim.env.unwrapped.vehicle.speed

    sim.render()
    time.sleep(0.1) 
    
    state_idx = next_state_idx
    step_count += 1

# Episode Statistics

speed /= step_count if step_count > 0 else 1.0
sim.close()


# Final Statistics
print("Simulation Episode Completed.")
print(f"Total Steps: {step_count}")
print(f"Average Speed: {speed:.2f} m/s")
print("\nSimulation Complete")


Simulation Episode Completed.
Total Steps: 1
Average Speed: 13.60 m/s

Simulation Complete
